Saving and Loading Results
==========================

Most of Beam Corset's :py:func:`~dataclasses.dataclass` based objects can be serialized to and from YAML to save data for later use.

Beam Corset Objects
-------------------

All objects that can be serialized to YAML implement two methods: :meth:`~corset.serialize.save_yaml` and :meth:`~corset.serialize.load_yaml`. The former is called on an instance of the object, taking a path to a file to write the YAML representation of the object. The latter is a class method that takes a path to a YAML file and returns an instance of the object.

In [ ]:
import os
from tempfile import TemporaryDirectory

tempdir = TemporaryDirectory()
prev_cwd = os.getcwd()
os.chdir(tempdir.name)

In [ ]:
from corset import Beam

beam = Beam.from_gauss(focus=200e-3, waist=100e-6, wavelength=1064e-9)
print("original:", beam)
beam.save_yaml("beam.yaml") # save through the instance method
loaded_beam = Beam.load_yaml("beam.yaml") # load through the class method
print("reloaded:", loaded_beam)

This also works for more complex objects like solution lists.

In [ ]:
from corset import ShiftingRange, SolutionList, ThinLens, mode_match

solutions = mode_match(Beam.from_gauss(0.0, 500e-6, 1064e-9), Beam.from_gauss(1.0, 100e-6, 1064e-9), [ShiftingRange(0.0, 0.8)], [ThinLens(f) for f in [100e-3, 150e-3]], 2, 2)
solutions.save_yaml("solutions.yaml") # save through the instance method
reloaded_solutions = SolutionList.load_yaml("solutions.yaml") # load through the class method
reloaded_solutions

Since most Beam Corset objects also include references to the data they are computed from, saving a solution will also save the underlying :class:`~corset.solver.ModeMatchingProblem` and all its relevant members.

In [ ]:
print(reloaded_solutions[0].candidate.problem.setup.initial_beam)

This is implemented fairly efficient since YAML allows emitting references to already serialized objects, so that the same object is not serialized multiple times.

Built-in Data Structures
------------------------

Beam Corset also supports serializing data structures containing beam corset objects, such as lists, tuples, and dictionaries. Everything that does not implement a :meth:`~corset.serialize.save_yaml` or :meth:`~corset.serialize.load_yaml` (and everything that does) can be serialized using the free :func:`~corset.serialize.save_yaml` and :func:`~corset.serialize.load_yaml` functions.

In [ ]:
from corset import load_yaml, save_yaml

my_dict = {
    "beam": beam,
    "other_data": [1, 2, 3],
}
save_yaml(my_dict, "my_dict.yaml") # save through the free function
reloaded_dict = load_yaml("my_dict.yaml") # load through the free function
reloaded_dict

Unlike the subclass methods for loading which only allow loading objects of the classes type. The free :func:`~corset.serialize.load_yaml` function can load any object from a YAML file, and will return the correct type of object based on the contents of the file.

PNG Files with Metadata
-----------------------

All beam corset objects that have an IPython PNG representation (i.e. they implement `_repr_png_`) can also be saved to PNG files with the YAML representation of the object embedded in the PNG metadata. This is done through the :meth:`~corset.serialize.YamlPngSerializableMixin.save_png` and :meth:`~corset.serialize.YamlPngSerializableMixin.load_png` methods.

This makes it very convenient to save Beam Corset's visualizations for viewing outside of Jupyter notebooks, while still preserving all underlying information for later analysis.

In [ ]:
from corset import ModeMatchingSolution

solutions[0].save_png("solution.png") # save through the instance method
reloaded_solution = ModeMatchingSolution.load_png("solution.png") # load through the class method
print("minimum coupling in reloaded solution:", reloaded_solution.analysis.min_coupling)
reloaded_solution

To make things even more convenient, the HTML representation of mode matching solutions also has two buttons that appear when hovering over the solution (they are also available in this documentation). They will save the solution as a YAML file or as a PNG file with the YAML representation embedded in the metadata respectively.

This is especially useful to immediately save a chosen solution while browsing through the visualizations of all solutions in a solution list to avoid any index based selection ambiguities.

In [ ]:
os.chdir(prev_cwd)
tempdir.cleanup()